RoPE（Rotary Position Embedding，旋转位置编码）是目前大语言模型（如 Llama, Mistral, PaLM 等）中最主流的位置编码技术。它由苏剑林等人在 2021 年提出，巧妙地结合了**绝对位置编码**的实现便利性和**相对位置编码**的优良特性。

---

## 核心思想：旋转即位置

传统的绝对位置编码（如 BERT）是直接在词向量上**加**一个位置向量。而 RoPE 的核心思路是：通过**旋转**向量来注入位置信息。

想象在一个二维平面上，有一个向量 $q$。如果我们想表示这个向量处于位置 $m$，我们就把它旋转角度 $m\theta$。如果另一个向量 $k$ 处于位置 $n$，我们就把它旋转 $n\theta$。

**为什么这样做能体现相对位置？**
当你计算 $q$ 和 $k$ 的点积时，结果只取决于它们之间的**夹角差** $(m-n)\theta$。这意味着模型在计算注意力时，能够自然地感知到两个词之间的相对距离。



---

## 数学原理推导

RoPE 的目标是找到一个变换 $f(x, m)$，使得两个向量经过变换后的内积满足：
$$\langle f(q, m), f(k, n) \rangle = g(q, k, m-n)$$

### 1. 二维情况
对于一个二维向量 $x = [x_1, x_2]^T$，RoPE 将其看作复数平面上的一个点。位置 $m$ 的旋转变换可以表示为乘以一个复数 $e^{im\theta}$：
$$f(x, m) = \begin{pmatrix} \cos m\theta & -\sin m\theta \\ \sin m\theta & \cos m\theta \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix}$$

### 2. 多维情况
对于高维词嵌入（维度为 $d$），RoPE 将维度两两分组，每一组都应用不同频率 $\theta_i$ 的旋转：
$$\theta_i = 10000^{-2i/d}, \quad i \in [0, 1, \dots, d/2-1]$$
最终的变换矩阵是一个分块对角矩阵。

### 3. RoPE 的优势
* **外推性**：虽然训练时长度有限，但由于旋转的周期性，理论上比绝对位置编码有更好的长度扩展潜力。
* **相对性**：点积结果严格随相对距离衰减（由于 $\theta$ 的设计），符合自然语言中“近距离词关系更紧密”的直觉。

---

## PyTorch 代码实现

在实际代码中，我们不会真的去乘巨大的旋转矩阵，而是利用复数运算或实数范围内的等价变换（计算优化版）来实现。



### 代码要点说明：
1.  **`inv_freq`**: 预计算频率。维度越高，旋转越慢（周期越长）。
2.  **`rotate_half`**: 这是一个巧妙的技巧。直接根据旋转矩阵公式 $x_1\cos - x_2\sin$ 和 $x_1\sin + x_2\cos$ 展开，可以避免显式构建稀疏矩阵，极大提高速度。
3.  **广播机制**: `cos` 和 `sin` 的形状通过 `None` 扩展，使其能自动作用于每一个 batch 和每一个 head。

In [ ]:
import torch
import torch.nn


def rotate_half(x: torch.Tensor):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000):
        super().__init__()

        theta_i = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))

        t = torch.arange(max_seq_len).type_as(theta_i)

        freqs = torch.einsum("i,j->ij", t, theta_i)

        emb = torch.cat([freqs, freqs], dim=-1)

        cos_cache = emb.cos()[None, None, :, :]
        sin_cache = emb.sin()[None, None, :, :]

        self.register_buffer("cos_cache", cos_cache)
        self.register_buffer("sin_cache", sin_cache)

    def forward(self, q: torch.Tensor, k: torch.Tensor):
        cos = self.cos_cache[:, :, : q.size(2), :]
        sin = self.sin_cache[:, :, : q.size(2), :]

        q_embed = q * cos + (rotate_half(q) * sin)
        k_embed = k * cos + (rotate_half(k) * sin)

        return q_embed, k_embed


# --- 使用示例 ---
batch, heads, seq_len, d_head = 2, 8, 128, 64
q = torch.randn(batch, heads, seq_len, d_head)
k = torch.randn(batch, heads, seq_len, d_head)

rope = RoPE(d_head)
q_rot, k_rot = rope(q, k)

print(f"Input shape: {q.shape}")
print(f"Output shape: {q_rot.shape}")

Input shape: torch.Size([2, 8, 128, 64])
Output shape: torch.Size([2, 8, 128, 64])




---

### 📝 笔记：位置编码维度与机制差异对比

#### 1. 核心差异概览
| 特性 | 常规位置编码 (如 Sinusoidal / Learnable) | 旋转位置编码 (RoPE) |
| :--- | :--- | :--- |
| **作用维度层级** | `d_model` (Embedding 全局维度) | `head_dim` (每个 Attention 头的内部维度) |
| **结合方式** | **相加 (Additive)**：$x + pos$ | **相乘/旋转 (Multiplicative)**：$q \cdot R$ |
| **作用时机** | 模型最底层，进入 Transformer 层之前 | 每个 Attention 层内，$Q$ 和 $K$ 变换之后 |
| **数学本质** | 赋予每个 Token 一个绝对坐标 | 通过旋转变换实现**相对位置**感应 |

---

#### 2. 为什么常规编码在 `d_model` 层级？
* **全局注入**：常规编码（如 BERT 或 GPT-2）是在输入层直接加到词向量上的。
* **线性传播**：因为 $Q = X \cdot W_q$，当位置信息已经存在于 $X$ 中时，经过矩阵乘法 $W_q$，位置信息会自动按权重分配到各个 Head 中。
* **逻辑**：它像是在一张大纸上画好坐标轴，然后再把纸裁成小条（Heads），每个小条自然带有了坐标信息。



---

#### 3. 为什么 RoPE 必须深入到 `head_dim`？
* **正交性保护**：RoPE 的数学原理是利用复数旋转保持内积的相对性。如果先旋转再乘 $W_q$ 矩阵，矩阵变换会破坏旋转的几何结构，导致相对位置关系失效。
* **头内独立计算**：
    1.  先分头：将 `d_model` 拆分为 `num_heads` 个 `head_dim`。
    2.  再旋转：在每个头对应的低维空间执行旋转。
* **计算效率**：在 `head_dim` 层级操作可以利用 PyTorch 的广播机制，所有头共用一组旋转参数（$cos/sin$），极大节省显存。
* **逻辑**：它是先裁好纸条（Heads），再在每个纸条上安装一个独立的指南针（Rotation）。



---

#### 4. 形象比喻
* **常规编码（绝对位置）**：给每个学生发一件印有座位号的 **T恤**（`d_model` 维度），学生进教室后无论怎么分组（Split Heads），座位号始终跟着他。
* **RoPE（旋转位置）**：学生进教室并按小组坐好后，要求每个组员根据自己的编号按特定角度 **转身**（`head_dim` 维度）。这样当两个组员对视时，他们旋转的角度差就代表了他们的相对距离。

---

**结论**：如果你在写代码或阅读源码（如 Llama, Baichuan），看到 RoPE 的输入形状是 `[batch, heads, seq, head_dim]` 而不是 `[seq, d_model]`，这是为了确保位置旋转能精确作用于注意力机制的内积运算中。


位置信息是为了帮助 $Q$ 找到正确的 $K$（即算准注意力分数）。一旦分数算好了，$V$ 只需要提供纯粹的语义内容即可。如果把 $V$ 也旋转了，反而会干扰最终输出的语义表达。